# Example 5 Business: Recovery of Alameda Island businesses using R2D damage estimates and third-party infrastructure simulators.

This notebook runs the Alameda Island business resilience simulation. It extends Example 5 with business impact modelling, tracking revenue losses caused by building damage, infrastructure outages, employee availability, local supplier access, and customer base disruption.

Example 5 shows how **pyrecodes** extends NHERI R2D's damage assessment to simulate recovery and integrate third-party infrastructure simulators of water supply systems and transportation systems to assess their interdependencies. Sparse distribution time stepping is used. 

Please refer to the **pyrecodes** [Example 5 page](https://nikolablagojevic.github.io/pyrecodes/html/usage/examples/example_5.html) for further details.

In [ ]:
import folium
import shapely
import pyproj
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from pyrecodes import main
from pyrecodes.component.r2d_component import R2DRoadway, R2DBridge, R2DTunnel, R2DBuildingWithBusiness
from pyrecodes.resource_distribution_model.residual_demand_traffic_distribution_model import ResidualDemandTrafficDistributionModel


def get_point_square(point_geom, half_side_m):
    """Create a square polygon centered on a point geometry with a given half-side in meters."""
    transformer_to_utm = pyproj.Transformer.from_crs("epsg:4326", "epsg:32610", always_xy=True)
    transformer_to_wgs = pyproj.Transformer.from_crs("epsg:32610", "epsg:4326", always_xy=True)
    utm_x, utm_y = transformer_to_utm.transform(point_geom.x, point_geom.y)
    utm_square = shapely.box(utm_x - half_side_m, utm_y - half_side_m,
                             utm_x + half_side_m, utm_y + half_side_m)
    wgs_coords = [transformer_to_wgs.transform(*coord) for coord in utm_square.exterior.coords]
    return wgs_coords

def get_bridge_square(component):
    """Create a square polygon centered on the bridge's point geometry, sized by deck width."""
    point = shapely.from_wkt(component.geometry)
    half_side_m = component.deck_width * 0.3048 / 2
    return get_point_square(point, half_side_m)

def get_tunnel_square(component):
    """Create a circular polygon centered on the tunnel's point geometry, sized by a fixed radius."""
    point = shapely.from_wkt(component.geometry)
    half_side_m = 150
    return get_point_square(point, half_side_m)

def get_traffic_nodes(system):
    """Extract traffic node locations from the ResidualDemandTrafficDistributionModel's flow_simulator.nodes_df."""
    for resource_name, resource_data in system.resources.items():
        dist_model = resource_data.get('DistributionModel', None)
        if isinstance(dist_model, ResidualDemandTrafficDistributionModel):
            nodes_df = dist_model.flow_simulator.nodes_df
            # nodes_df has columns: node_id, x (lon), y (lat)
            return [(row['node_id'], row['y'], row['x']) for _, row in nodes_df.iterrows()]
    return []


def build_map(snapshot, time_step):
    """Build a folium map from a saved snapshot of component states."""
    roads = snapshot['roads']
    bridges = snapshot['bridges']
    tunnels = snapshot['tunnels']
    buildings = snapshot.get('buildings', [])
    traffic_nodes = snapshot.get('traffic_nodes', [])
    center = snapshot['center']
    center_lat, center_lon = center
    m = folium.Map(location=[center_lat, center_lon], zoom_start=14)

    for name, coords, func_level in roads:
        color = 'green' if func_level >= 1.0 else 'red'
        folium.PolyLine(
            coords, color=color, weight=4, opacity=0.8,
            tooltip=f"{name} | Functionality: {func_level:.2f}"
        ).add_to(m)

    for name, folium_coords, func_level in bridges:
        color = 'green' if func_level >= 1.0 else 'red'
        folium.Polygon(
            folium_coords, color=color, fill=True, fill_color=color,
            fill_opacity=0.6, weight=2,
            tooltip=f"{name} | Functionality: {func_level:.2f}"
        ).add_to(m)

    for name, folium_coords, func_level in tunnels:
        color = 'green' if func_level >= 1.0 else 'red'
        folium.Polygon(
            folium_coords, color=color, fill=True, fill_color=color,
            fill_opacity=0.6, weight=2,
            tooltip=f"{name} | Functionality: {func_level:.2f}"
        ).add_to(m)

    for node_id, lat, lon in traffic_nodes:
        folium.CircleMarker(
            location=[lat, lon], radius=3,
            color='gray', fill=True, fill_color='gray',
            fill_opacity=0.5, weight=1,
            tooltip=f"Traffic node {node_id}"
        ).add_to(m)

    for building_name, footprint_coords, tooltip_text in buildings:
        folium.Polygon(
            footprint_coords, color='blue', fill=True, fill_color='blue',
            fill_opacity=0.4, weight=1,
            tooltip=folium.Tooltip(tooltip_text, sticky=True)
        ).add_to(m)

    folium.map.Marker(
        [center_lat, center_lon],
        icon=folium.DivIcon(
            html=f'<div style="font-size:14px;font-weight:bold;background:white;padding:4px;border-radius:4px;">Time step: {time_step}</div>'
        )
    ).add_to(m)
    return m


def capture_snapshot(system, time_step, traffic_nodes):
    """Capture current road/bridge/tunnel/building state as lightweight data for later rendering."""
    road_components = [c for c in system.components if isinstance(c, R2DRoadway)]
    bridge_components = [c for c in system.components if isinstance(c, R2DBridge)]
    tunnel_components = [c for c in system.components if isinstance(c, R2DTunnel)]
    business_buildings = [c for c in system.components if isinstance(c, R2DBuildingWithBusiness)]

    if not road_components and not bridge_components and not tunnel_components:
        return None

    first_comp = road_components[0] if road_components else (bridge_components[0] if bridge_components else tunnel_components[0])
    first_geom = shapely.from_wkt(first_comp.geometry)
    center = (first_geom.centroid.y, first_geom.centroid.x)

    roads = []
    for c in road_components:
        line = shapely.from_wkt(c.geometry)
        coords = [(lat, lon) for lon, lat in line.coords]
        roads.append((c.name, coords, c.functionality_level))

    bridges = []
    for c in bridge_components:
        square_coords = get_bridge_square(c)
        folium_coords = [(lat, lon) for lon, lat in square_coords]
        bridges.append((c.name, folium_coords, c.functionality_level))

    tunnels = []
    for c in tunnel_components:
        square_coords = get_tunnel_square(c)
        folium_coords = [(lat, lon) for lon, lat in square_coords]
        tunnels.append((c.name, folium_coords, c.functionality_level))

    buildings = []
    for c in business_buildings:
        if hasattr(c, 'footprint') and hasattr(c, 'businesses'):
            geojson_coords = c.footprint['geometry']['coordinates'][0]
            footprint_coords = [(lat, lon) for lon, lat in geojson_coords]
            tooltip_lines = [f"<b>{c.name}</b><br>"]
            for biz in c.businesses:
                reasons = biz.reason_for_drop.get(time_step, [])
                if reasons:
                    reason_strs = [f"{r['Name']}: {r['Level']:.2f}" for r in reasons]
                    tooltip_lines.append(f"Business {biz.business_id}: {', '.join(reason_strs)}")
                else:
                    tooltip_lines.append(f"Business {biz.business_id}: No drop")
            tooltip_text = '<br>'.join(tooltip_lines)
            buildings.append((c.name, footprint_coords, tooltip_text))

    return {
        'roads': roads, 'bridges': bridges, 'tunnels': tunnels,
        'buildings': buildings, 'traffic_nodes': traffic_nodes, 'center': center
    }


def show_slider(snapshots):
    """Display an interactive slider to browse maps across all recorded time steps."""
    time_steps = sorted(snapshots.keys())
    if not time_steps:
        return

    map_output = widgets.Output()
    slider = widgets.IntSlider(
        value=time_steps[-1], min=time_steps[0], max=time_steps[-1],
        step=1, description='Time step:', continuous_update=False,
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='80%')
    )

    def on_slider_change(change):
        ts = change['new']
        nearest = min(time_steps, key=lambda t: abs(t - ts))
        with map_output:
            clear_output(wait=True)
            m = build_map(snapshots[nearest], nearest)
            display(m)

    slider.observe(on_slider_change, names='value')

    with map_output:
        m = build_map(snapshots[time_steps[-1]], time_steps[-1])
        display(m)

    display(widgets.VBox([slider, map_output]))


def run_with_road_plots(config_file):
    """Run pyrecodes simulation and show a post-simulation slider with spatial plots."""
    input_dict = main.read_json_file(config_file)
    system = main.create_system(input_dict)
    snapshots = {}

    # Extract traffic nodes once (they don't change during simulation)
    traffic_nodes = get_traffic_nodes(system)

    for system.time_step in range(system.START_TIME_STEP, system.MAX_TIME_STEP):
        print(f"Time step: {system.time_step}")

        if system.recovery_target_met():
            system.FINISH = True

        if system.time_step == system.DISASTER_TIME_STEP:
            system.set_initial_damage()

        system.update()
        system.distribute_resources()
        system.update_resilience_calculators()

        snapshot = capture_snapshot(system, system.time_step, traffic_nodes)
        if snapshot is not None:
            snapshots[system.time_step] = snapshot

        if system.time_step > system.DISASTER_TIME_STEP:
            system.recover()

        if system.FINISH:
            print('Resilience assessment finished.')
            break

    # Show interactive slider after simulation completes
    show_slider(snapshots)

    system.road_network_snapshots = snapshots
    return system

In [ ]:
system = run_with_road_plots('./Example 5_business/Alameda_Main.json')

system.calculate_resilience()

In [ ]:
import random

from pyrecodes.plotter.business_plotter import BusinessPlotter

business_plotter = BusinessPlotter()
business_resilience_calculator = system.resilience_calculators[-1]

for business_to_plot in random.sample(business_resilience_calculator.businesses, 5):
    reasons_for_drop = business_plotter.get_reasons_for_drop(business_to_plot)
    reasons_as_lines = business_plotter.get_reasons_for_drop_as_lines(reasons_for_drop)
    # business_plotter.plot_business_revenue(business_resilience_calculator.business_revenue[business_to_plot], business_to_plot, save_fig=False, show_fig=True)
    # business_plotter.plot_business_revenue_reasons_for_drop_lines(business_to_plot, reasons_as_lines, reasons_to_plot=['Home Component Functionality'], save_fig=False, show_fig=True)
    # business_plotter.plot_business_revenue_reasons_for_drop_lines(business_to_plot, reasons_as_lines, reasons_to_plot=['Home Component Functionality', 'Customer Base'], save_fig=False, show_fig=True)
    # business_plotter.plot_business_revenue_reasons_for_drop_lines(business_to_plot, reasons_as_lines, reasons_to_plot=['Home Component Functionality', 'Customer Base', 'LocalSuppliers'], save_fig=False, show_fig=True)
    # business_plotter.plot_business_revenue_reasons_for_drop_lines(business_to_plot, reasons_as_lines, reasons_to_plot=['Home Component Functionality', 'Customer Base', 'LocalSuppliers', 'Labor'], save_fig=False, show_fig=True)
    # business_plotter.plot_business_revenue_reasons_for_drop_lines(business_to_plot, reasons_as_lines, reasons_to_plot=['Home Component Functionality', 'Customer Base', 'LocalSuppliers', 'Labor', 'Infrastructure'], save_fig=False, show_fig=True)
    business_plotter.plot_business_revenue_reasons_for_drop_lines(business_to_plot, reasons_as_lines, save_fig=False, show_fig=True)
    business_plotter.plot_business_gantt_chart(business_to_plot, save_fig=False, show_fig=True)

In [ ]:
total_revenue = business_plotter.calculate_total_revenue(business_resilience_calculator)
total_revenue_no_building_damage = business_plotter.calculate_total_revenue_no_building_damage(business_resilience_calculator)
business_plotter.plot_total_revenue(total_revenue, show_fig=True, save_fig=False)
business_plotter.plot_total_revenue_BI_CBI(total_revenue, total_revenue_no_building_damage, show_fig=True, save_fig=False)


In [ ]:
total_reasons_for_drop = business_plotter.get_total_reasons_for_drop(business_resilience_calculator.businesses)
total_reasons_as_lines = business_plotter.get_total_reasons_for_drop_as_lines(total_reasons_for_drop)

business_plotter.plot_total_revenue_reasons_for_drop_lines(total_reasons_as_lines, save_fig=False, show_fig=True)

In [ ]:
business_plotter.plot_lost_revenue_to_repair_cost_histogram(
    business_resilience_calculator.lost_revenue_to_repair_cost_ratio, bins=50, show_fig=True, save_fig=False
)